## Load Data

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

job_folder = Path("../../data/job")

def clean_html(text):
    if not text:
        return ""
    text = str(text)
    soup = BeautifulSoup(text, "html.parser")
    cleaned = soup.get_text(separator=" ")
    cleaned = cleaned.replace("\xa0", " ")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

rows = []

for file_path in job_folder.glob("*.json"):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            job = json.load(f)

        skills_list = [
            s.get("skill")
            for s in job.get("skills", [])
            if isinstance(s, dict) and s.get("skill")
        ]

        flexible_work_arrangements = [
            fwa.get("flexibleWorkArrangement") or fwa.get("arrangement") or fwa.get("name")
            for fwa in job.get("flexibleWorkArrangements", [])
            if isinstance(fwa, dict)
        ]

        employment_types = [
            e.get("employmentType")
            for e in job.get("employmentTypes", [])
            if isinstance(e, dict) and e.get("employmentType")
        ]

        position_levels = [
            p.get("position")
            for p in job.get("positionLevels", [])
            if isinstance(p, dict) and p.get("position")
        ]

        salary_info = job.get("salary") or {}
        salary_type = (salary_info.get("type") or {}).get("salaryType")

        metadata = job.get("metadata") or {}

        row = {
            "uuid": job.get("uuid"),
            "title": job.get("title"),
            "description": clean_html(job.get("description")),
            "minimum_years_experience": job.get("minimumYearsExperience"),
            "skills": skills_list,

            "ssoc_code": job.get("ssocCode"),
            "ssoc_version": job.get("ssocVersion"),

            "salary_minimum": salary_info.get("minimum"),
            "salary_maximum": salary_info.get("maximum"),
            "salary_type": salary_type,

            "employment_types": employment_types,
            "position_levels": position_levels,
            "flexible_work_arrangements": flexible_work_arrangements,

            "posting_date": metadata.get("newPostingDate") or metadata.get("originalPostingDate"),
            "expiry_date": metadata.get("expiryDate"),
        }

        rows.append(row)

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")



original_jobs_df = pd.DataFrame(rows)

print("Original shape:", original_jobs_df.shape)
original_jobs_df.head()

Original shape: (22720, 15)


# Cleaning

## Create copy of jobs_df

In [207]:
jobs_df = original_jobs_df.copy()
jobs_df.head(1)

,uuid,title,description,minimum_years_experience,skills,ssoc_code,ssoc_version,salary_minimum,salary_maximum,salary_type,employment_types,position_levels,flexible_work_arrangements,posting_date,expiry_date
0,2cc5117f205f59b400cb529d2253651a,Outdoor Sales Engineer (Own Car) | Petrochemic...,Role: Outdoor Sales Engineer Working Hours: 8....,1,"[Project Bidding, Technical Demonstrations, Sh...",24331,2020v2,3000,6000,Monthly,"[Permanent, Full Time]",[Executive],[],2026-01-29,2026-02-28


In [208]:
# Only keep years 0 and 1
jobs_df = jobs_df[jobs_df["minimum_years_experience"].isin([0, 1])]

## Removing interns/ further cleaning

In [209]:
# -------------------------
# 1. lowercase
# -------------------------
jobs_df["title_clean"] = jobs_df["title"].str.lower()
jobs_df["description_clean"] = jobs_df["description"].str.lower()


# -------------------------
# 2. internship filter
# -------------------------
intern_mask = (
    # title / description
    jobs_df["title_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False) |
    jobs_df["description_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False)
)


# -------------------------
# 3. employment_types filter
# -------------------------
employment_mask = jobs_df["employment_types"].apply(
    lambda x: isinstance(x, list) and any(
        "intern" in str(e).lower() for e in x
    )
)


# -------------------------
# 4. combine + filter
# -------------------------
jobs_df = jobs_df[~(intern_mask | employment_mask)]

/var/folders/gn/g9t9ks6j5qq32pcsbtb27jxr0000gn/T/ipykernel_2597/3693494163.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jobs_df["title_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False) |
/var/folders/gn/g9t9ks6j5qq32pcsbtb27jxr0000gn/T/ipykernel_2597/3693494163.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jobs_df["description_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False)


In [210]:
# Create a True/ False for Flexible Work Arrangments
jobs_df["has_flexible_work"] = jobs_df["flexible_work_arrangements"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

In [211]:
jobs_df = jobs_df.drop(columns=["flexible_work_arrangements"], errors="ignore")

In [212]:
jobs_df["ssoc_3d"] = jobs_df["ssoc_code"].astype(str).str[:3]

In [213]:
jobs_df = jobs_df.drop(columns=["ssoc_code"])
jobs_df = jobs_df.drop(columns=["ssoc_version"])
jobs_df = jobs_df.drop(columns=["description"])
jobs_df = jobs_df.drop(columns=["description_clean"])
jobs_df = jobs_df.drop(columns=["title"])
jobs_df = jobs_df.drop(columns=["title_clean"])
jobs_df = jobs_df.drop(columns=["position_levels"])
jobs_df = jobs_df.drop(columns=["minimum_years_experience"])
jobs_df = jobs_df.drop(columns=["salary_type"])

## Employment Type

In [214]:
def clean_skill(skill):
    if not isinstance(skill, str):
        return None

    skill = skill.lower().strip()

    replacements = {
        "mgmt": "management",
        "&": "and"
    }

    for k, v in replacements.items():
        skill = skill.replace(k, v)

    return skill

jobs_df["skills_clean"] = jobs_df["skills"].apply(
    lambda lst: [clean_skill(s) for s in lst if clean_skill(s)]
)

jobs_df = jobs_df.drop(columns=["skills"])

In [215]:
contract_types = [
    "Permanent", "Contract", "Temporary", "Freelance"
]

work_types = [
    "Full Time", "Part Time"
]

def extract_contract_type(lst):
    if isinstance(lst, list):
        for item in lst:
            if item in contract_types:
                return item
    return None

def extract_work_type(lst):
    if isinstance(lst, list):
        for item in lst:
            if item in work_types:
                return item
    return None

jobs_df["contract_type"] = jobs_df["employment_types"].apply(extract_contract_type)
jobs_df["work_type"] = jobs_df["employment_types"].apply(extract_work_type)

jobs_df = jobs_df.drop(columns=["employment_types"])

In [216]:
jobs_df["contract_type"] = jobs_df["contract_type"].fillna("Unknown")
jobs_df["work_type"] = jobs_df["work_type"].fillna("Unknown")

In [217]:
jobs_df["work_type"].value_counts()

work_type
Full Time    6894
Unknown      1287
Part Time     766
Name: count, dtype: int64

In [218]:
jobs_df["contract_type"].value_counts()

contract_type
Unknown      4079
Permanent    3557
Contract      882
Temporary     361
Freelance      68
Name: count, dtype: int64

In [219]:
jobs_df.head(1)

,uuid,salary_minimum,salary_maximum,posting_date,expiry_date,has_flexible_work,ssoc_3d,skills_clean,contract_type,work_type
0,2cc5117f205f59b400cb529d2253651a,3000,6000,2026-01-29,2026-02-28,False,243,"[project bidding, technical demonstrations, sh...",Permanent,Full Time


In [220]:
output_path = Path("../../data/jobs_cleaned.xlsx")

jobs_df.to_excel(output_path, index=False)